<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_23_property_decorators_dunder/note_lesson_23_property_dunder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 23 — `@property`, декоратори класів, dunder

Мобільний застосунок сервісу «Смачно + Таксі» пише звичайний Python поверх наших класів: `sorted(deliveries)`, `len(cart)`, `"Борщ" in cart`, `price + fee`. Сьогодні вчимо класи працювати з цим синтаксисом:

1. **Dunder-методи** — протоколи Python: контейнер, арифметика, порівняння, хеш, виклик.
2. **`@property` і дескриптори** — обчислювані атрибути й одна перевірка на багато полів.
3. **Декоратори класів** — реєстр, `@total_ordering`, `@dataclass`.

Виконуй клітинки **зверху вниз**; перед **Прогнозом** спершу скажи, що буде. Теорія, схеми й архітектура — у книзі: [Урок 23. @property, декоратори класів, dunder](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_23/).

## 🔁 Пригадай (без підглядання)

1. Що приймає і що повертає декоратор функції (урок 9)?
2. Яке правило пов'язує хеш і рівність ключів словника (урок 16)?
3. Що робить `@property` без сеттера (урок 21)?

<details>
<summary>Відповіді</summary>

1. Приймає функцію й повертає функцію.
2. Рівні ключі мусять мати однаковий хеш.
3. Дає читати метод як атрибут; запис падає з `AttributeError`.

</details>

## 1. Dunder-методи: протоколи Python

**Прогноз:** що буде з `sorted(deliveries)` і чи рівні дві однакові доставки?

In [ ]:
class Delivery:
    def __init__(self, order_id, minutes):
        self.order_id = order_id
        self.minutes = minutes


deliveries = [Delivery(1, 35), Delivery(2, 20), Delivery(3, 50)]
try:
    sorted(deliveries)
except TypeError as error:
    print(error)
print(Delivery(1, 35) == Delivery(1, 35))

<details>
<summary>Відповідь</summary>

`TypeError`: Python не знає, що означає «менша доставка». А `==` за замовчуванням порівнює ідентичність — це два різні об'єкти.

</details>

In [ ]:
class Cart:
    def __init__(self):
        self._items = {}

    def add(self, dish, qty=1):
        self._items[dish] = self._items.get(dish, 0) + qty

    def __len__(self):
        return sum(self._items.values())

    def __contains__(self, dish):
        return dish in self._items

    def __iter__(self):
        return iter(self._items.items())

    def __getitem__(self, dish):
        return self._items[dish]

    def __repr__(self):
        return f"Cart({self._items})"


cart = Cart()
cart.add("Борщ", 2)
cart.add("Узвар")
print(len(cart), "Борщ" in cart, cart["Борщ"])
for dish, qty in cart:
    print(dish, qty)

**Прогноз:** методу `__bool__` немає. Що надрукує `bool(Cart())`?

In [ ]:
print(bool(Cart()), bool(cart))

<details>
<summary>Відповідь</summary>

`False True`: без `__bool__` Python бере `__len__`, нуль — хибність.

</details>

### Гроші: арифметика й `NotImplemented`

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.amount + other.amount)

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount

    def __repr__(self):
        return f"Money({self.amount})"

    def __str__(self):
        return f"{self.amount} грн"


bill = Money(95) + Money(60)
print(bill, repr(bill), bill == Money(155))
try:
    Money(95) + 60
except TypeError as error:
    print(error)

**Прогноз:** чи спрацює `sum(prices)` для списку `Money`?

In [ ]:
prices = [Money(95), Money(60), Money(40)]
try:
    sum(prices)
except TypeError as error:
    print(error)
print(sum(prices, Money(0)))

<details>
<summary>Відповідь</summary>

Ні: `sum` починає з `0`, а `0 + Money(95)` не вміє ні `int`, ні `Money` (немає `__radd__`).

</details>

In [ ]:
class Money(Money):
    def __radd__(self, other):
        if other == 0:
            return self
        return NotImplemented


print(sum([Money(95), Money(60), Money(40)]))

### `__eq__` і `__hash__`

**Прогноз:** чи можна покласти два `Money(95)` у множину?

In [ ]:
try:
    {Money(95), Money(95)}
except TypeError as error:
    print(error)
print(Money.__hash__)

<details>
<summary>Відповідь</summary>

Ні: визначивши `__eq__`, клас втратив `__hash__` (він став `None`).

</details>

In [ ]:
class Money(Money):
    def __hash__(self):
        return hash(self.amount)


print(len({Money(95), Money(95), Money(60)}), Money(95) in {Money(95)})

**Прогноз:** монету змінили після того, як поклали в множину. Чи знайде її `in`?

In [ ]:
wallet = {Money(95)}
coin = next(iter(wallet))
coin.amount = 100
print(Money(100) in wallet, coin in wallet)

<details>
<summary>Відповідь</summary>

`False False`: монета лежить у комірці старого хешу. Хешованим має бути лише незмінний об'єкт.

</details>

### Порівняння і сортування

**Прогноз:** є лише `__lt__`. Що буде з `sorted`, `max` і `<=`?

In [ ]:
class Delivery:
    def __init__(self, order_id, minutes):
        self.order_id = order_id
        self.minutes = minutes

    def __lt__(self, other):
        return self.minutes < other.minutes

    def __repr__(self):
        return f"Delivery(№{self.order_id}, {self.minutes} хв)"


deliveries = [Delivery(1, 35), Delivery(2, 20), Delivery(3, 50)]
print(sorted(deliveries))
print(max(deliveries))
try:
    Delivery(1, 35) <= Delivery(2, 20)
except TypeError as error:
    print(error)

<details>
<summary>Відповідь</summary>

`sorted` і `max` працюють (`a > b` Python перевертає в `b < a`), а для `<=` пари немає — `TypeError`.

</details>

In [ ]:
print(sorted(deliveries, key=lambda delivery: delivery.order_id, reverse=True))

## 🛠 Вправа 1. Меню як контейнер

Допиши `Menu`: `len(menu)` — кількість страв, `dish in menu`, `menu[dish]` — ціна, `for dish in menu` — назви **від найдешевшої**, `repr` — `Menu(страв: 4)`.

In [ ]:
class Menu:
    def __init__(self, prices):
        self._prices = dict(prices)

    # YOUR CODE HERE
    # BEGIN SOLUTION
    def __len__(self):
        return len(self._prices)

    def __contains__(self, dish):
        return dish in self._prices

    def __getitem__(self, dish):
        return self._prices[dish]

    def __iter__(self):
        return iter(sorted(self._prices, key=self._prices.get))

    def __repr__(self):
        return f"Menu(страв: {len(self)})"
    # END SOLUTION


menu = Menu({"Борщ": 95, "Вареники": 110, "Узвар": 40, "Хліб": 10})
print(menu, list(menu))
assert len(menu) == 4 and "Узвар" in menu and "Піца" not in menu
assert menu["Борщ"] == 95
assert list(menu) == ["Хліб", "Узвар", "Борщ", "Вареники"]
assert repr(menu) == "Menu(страв: 4)"
assert not Menu({})
print("✅ Вправа 1 пройдена")

### `__call__`: об'єкт як функція

In [ ]:
class Surge:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, fare):
        return round(fare * self.factor)


evening = Surge(1.5)
print(evening(120), callable(evening))
print(list(map(evening, [100, 80])))

## 2. `@property` докладніше: обчислювані атрибути

In [ ]:
PRICES = {"Борщ": 95, "Вареники": 110, "Узвар": 40}


class Cart(Cart):
    @property
    def total(self):
        return sum(PRICES[dish] * qty for dish, qty in self)


cart = Cart()
cart.add("Борщ", 2)
cart.add("Узвар")
print(cart.total)
cart.add("Вареники")
print(cart.total)

**Прогноз:** що станеться при створенні `Courier`?

In [ ]:
class Courier:
    def __init__(self, name, rating):
        self.name = name
        self.rating = rating

    @property
    def rating(self):
        return self.rating

    @rating.setter
    def rating(self, value):
        if not 1 <= value <= 5:
            raise ValueError("рейтинг має бути від 1 до 5")
        self.rating = value


try:
    Courier("Олег", 4.8)
except RecursionError as error:
    print(type(error).__name__)

<details>
<summary>Відповідь</summary>

`RecursionError`: `self.rating = value` у сеттері знову викликає сеттер. Зберігати треба в `self._rating`.

</details>

**Прогноз:** скільки разів надрукується «рахую маршрут…»?

In [ ]:
from functools import cached_property


class Route:
    def __init__(self, stops):
        self.stops = stops

    @cached_property
    def length(self):
        print("рахую маршрут…")
        return sum(abs(b - a) for a, b in zip(self.stops, self.stops[1:]))


route = Route([0, 4, 1, 7])
print(route.length)
print(route.length)

<details>
<summary>Відповідь</summary>

Один: результат ліг в атрибут екземпляра, і наступне читання знаходить його там.

</details>

## 3. Дескриптори: одна перевірка на багато полів

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.name = "_" + name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.name)

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name[1:]} має бути більшим за 0, а маємо {value}")
        setattr(obj, self.name, value)


class Tariff:
    base = Positive()
    per_km = Positive()
    min_fare = Positive()

    def __init__(self, base, per_km, min_fare):
        self.base = base
        self.per_km = per_km
        self.min_fare = min_fare

    def fare(self, km):
        return max(self.min_fare, self.base + self.per_km * km)


day = Tariff(40, 12, 80)
print(day.fare(2), day.fare(10))
try:
    day.per_km = -5
except ValueError as error:
    print(error)
print(vars(day))

In [ ]:
print(type(vars(Tariff)["base"]).__name__, hasattr(property, "__set__"))

## 🛠 Вправа 2. Дескриптор `Range`

`Range(low, high)` пропускає лише значення з `[low, high]`, інакше `ValueError`. Значення зберігай в `_<ім'я>`, як `Positive`.

In [ ]:
class Range:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def __init__(self, low, high):
        self.low = low
        self.high = high

    def __set_name__(self, owner, name):
        self.name = "_" + name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.name)

    def __set__(self, obj, value):
        if not self.low <= value <= self.high:
            raise ValueError(f"{self.name[1:]} має бути від {self.low} до {self.high}, а маємо {value}")
        setattr(obj, self.name, value)
    # END SOLUTION


class Courier:
    rating = Range(1, 5)
    experience = Range(0, 50)

    def __init__(self, name, rating, experience):
        self.name = name
        self.rating = rating
        self.experience = experience


oleh = Courier("Олег", 4.8, 3)
print(vars(oleh))
assert oleh.rating == 4.8 and isinstance(Courier.rating, Range)
assert vars(oleh) == {"name": "Олег", "_rating": 4.8, "_experience": 3}
for bad in [lambda: Courier("Ірина", 7, 2), lambda: Courier("Ірина", 4.5, -1)]:
    try:
        bad()
        raise AssertionError("мало впасти з ValueError")
    except ValueError as error:
        print(error)
couriers = [oleh, Courier("Ірина", 4.95, 5), Courier("Тарас", 4.2, 1)]
best = sorted(couriers, key=lambda courier: courier.rating, reverse=True)
assert [courier.name for courier in best] == ["Ірина", "Олег", "Тарас"]
print("✅ Вправа 2 пройдена")

## 4. Декоратори класів

Декоратор класу приймає клас і повертає клас.

In [ ]:
PAYMENTS = {}


def payment(code):
    def register(cls):
        PAYMENTS[code] = cls
        return cls
    return register


@payment("card")
class CardPayment:
    def pay(self, amount):
        return f"картка: {amount} грн"


@payment("cash")
class CashPayment:
    def pay(self, amount):
        return f"готівка кур'єру: {amount} грн"


print(PAYMENTS)
print(PAYMENTS["cash"]().pay(340))

In [ ]:
from functools import total_ordering


@total_ordering
class Money(Money):
    def __lt__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount < other.amount


print(Money(95) > Money(60), Money(60) <= Money(60), max([Money(95), Money(40)]))

## 🛠 Вправа 3. Реєстр знижок

Напиши декоратор `discount(code)`, що записує клас у `DISCOUNTS[code]` і **повертає клас без змін**.

In [ ]:
DISCOUNTS = {}


def discount(code):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def register(cls):
        DISCOUNTS[code] = cls
        return cls
    return register
    # END SOLUTION


@discount("student")
class StudentDiscount:
    def apply(self, amount):
        return round(amount * 0.9)


@discount("night")
class NightDiscount:
    def apply(self, amount):
        return max(amount - 50, 0)


print(DISCOUNTS)
assert set(DISCOUNTS) == {"student", "night"}
assert DISCOUNTS["student"]().apply(340) == 306
assert DISCOUNTS["night"]().apply(30) == 0
assert StudentDiscount.__name__ == "StudentDiscount", "декоратор має повернути клас"
print("✅ Вправа 3 пройдена")

### `@dataclass`

**Прогноз:** які з методів `__init__`, `__repr__`, `__eq__`, `__lt__`, `__hash__` згенерує `@dataclass(frozen=True, order=True)`?

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True, order=True)
class Money:
    amount: int

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.amount + other.amount)

    def __str__(self):
        return f"{self.amount} грн"


a = Money(95)
print(repr(a), a == Money(95), a < Money(100), len({a, Money(95)}))
print([name for name in ("__init__", "__repr__", "__eq__", "__lt__", "__hash__") if name in vars(Money)])
try:
    a.amount = 100
except AttributeError as error:
    print(type(error).__name__)

<details>
<summary>Відповідь</summary>

Усі п'ять. `order=True` дає порівняння, `frozen=True` — незмінність і `__hash__`.

</details>

## 🛠 Вправа 4. Відстань як об'єкт-значення

`Distance(km)` — незмінний датаклас з порядком: `+` між відстанями, `sum(legs)` без стартового значення, `str` → `"8.0 км"`, від'ємна відстань — `ValueError` у `__post_init__`.

In [ ]:
from dataclasses import dataclass

# YOUR CODE HERE
# BEGIN SOLUTION
@dataclass(frozen=True, order=True)
class Distance:
    km: float

    def __post_init__(self):
        if self.km < 0:
            raise ValueError(f"відстань не може бути від'ємною: {self.km}")

    def __add__(self, other):
        if not isinstance(other, Distance):
            return NotImplemented
        return Distance(self.km + other.km)

    def __radd__(self, other):
        if other == 0:
            return self
        return NotImplemented

    def __str__(self):
        return f"{self.km} км"
# END SOLUTION


legs = [Distance(2.5), Distance(4.0), Distance(1.5)]
print(sum(legs), max(legs))
assert sum(legs) == Distance(8.0) and str(sum(legs)) == "8.0 км"
assert max(legs) == Distance(4.0) and sorted(legs)[0] == Distance(1.5)
assert len({Distance(1.5), Distance(1.5)}) == 1
for bad in [lambda: Distance(-1), lambda: setattr(legs[0], "km", 10)]:
    try:
        bad()
        raise AssertionError("мало впасти")
    except (ValueError, AttributeError) as error:
        print(type(error).__name__)
print("✅ Вправа 4 пройдена")

## 5. Розібраний приклад: кошик із грошима

In [ ]:
from dataclasses import dataclass, field


@dataclass(frozen=True, order=True)
class Money:
    amount: int

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return Money(self.amount + other.amount)

    def __mul__(self, times):
        return Money(self.amount * times)

    def __str__(self):
        return f"{self.amount} грн"


@dataclass(frozen=True)
class CartItem:
    dish: str
    price: Money
    qty: int = 1

    def __post_init__(self):
        if self.qty < 1:
            raise ValueError(f"кількість має бути від 1, а маємо {self.qty}")

    @property
    def total(self):
        return self.price * self.qty


@dataclass
class Cart:
    items: list = field(default_factory=list)

    def add(self, dish, price, qty=1):
        self.items.append(CartItem(dish, Money(price), qty))

    def __len__(self):
        return sum(item.qty for item in self.items)

    def __iter__(self):
        return iter(self.items)

    def __contains__(self, dish):
        return any(item.dish == dish for item in self.items)

    @property
    def total(self):
        return sum((item.total for item in self.items), Money(0))


cart = Cart()
cart.add("Борщ", 95, 2)
cart.add("Узвар", 40)
print(len(cart), "Узвар" in cart, cart.total)
print(max(cart, key=lambda item: item.total))
try:
    cart.add("Вареники", 110, 0)
except ValueError as error:
    print(error)
print(len(cart))

## 🛠 Вправа 5. Об'єднати кошики

`family = mom + son` — **новий** кошик з усіма позиціями; вихідні кошики не змінюються. З не-кошиком — `NotImplemented`.

In [ ]:
class Cart(Cart):
    def __add__(self, other):
        # YOUR CODE HERE
        # BEGIN SOLUTION
        if not isinstance(other, Cart):
            return NotImplemented
        return Cart(self.items + other.items)
        # END SOLUTION


mom = Cart()
mom.add("Борщ", 95, 2)
son = Cart()
son.add("Деруни", 85)
family = mom + son
print(len(family), family.total)
assert len(family) == 3 and family.total == Money(275)
assert len(mom) == 2 and len(son) == 1 and family is not mom
try:
    mom + 5
    raise AssertionError("мало впасти з TypeError")
except TypeError:
    pass
print("✅ Вправа 5 пройдена")

## ✅ Самоперевірка

1. Що викликає Python для `if cart:`, якщо немає `__bool__`?
2. Навіщо `__add__` повертає `NotImplemented`?
3. Чому після `__eq__` об'єкт не лізе в `set`?
4. Чому в сеттері не можна писати `self.rating = value`?
5. Що робить `__set_name__`?
6. Чим об'єкт-значення відрізняється від сутності?

<details>
<summary>Відповіді</summary>

1. `__len__`.
2. Щоб Python спробував `__radd__` іншого операнда, а потім сам кинув `TypeError`.
3. Python прибирає `__hash__`, щоб рівні об'єкти не мали різних хешів.
4. Це знову виклик сеттера — `RecursionError`. Зберігати в `self._rating`.
5. Повідомляє дескриптору ім'я атрибута; викликається під час створення класу.
6. Значення рівні за полями й незмінні (`Money`); сутність має ідентичність і змінюється (`Order`).

</details>

### Шпаргалка

```python
def __len__(self): ...            # len(x), bool(x)
def __contains__(self, item): ... # item in x
def __iter__(self): ...           # for, list(x), max(x)
def __add__(self, other):         # a + b
    if not isinstance(other, Money):
        return NotImplemented
def __radd__(self, other): ...    # 0 + a  (sum)

class Positive:                   # дескриптор
    def __set_name__(self, owner, name): self.name = "_" + name
    def __get__(self, obj, objtype=None): ...
    def __set__(self, obj, value): ...

@dataclass(frozen=True, order=True)   # значення: __init__, __repr__, __eq__, <, __hash__
class Money:
    amount: int
    items: list = field(default_factory=list)   # не []!
```

## Далі

- **Практикум на реальних даних**: `lab_lesson_23_cars_descriptors.ipynb` — дескриптор `PositiveNumber` для автомобілів.
- **Урок 24 — Ітератори advanced**: власний `__next__`, `.send()`, конвеєри `itertools`.